# LLM-as-Judge: Biotech Lyric Battles

This notebook evaluates four frontier language models on creative songwriting tasks at the intersection of biotechnology and music.

The project has three layers:

1. Generate lyrics from four models across fourteen biotech/music prompts.
2. Use the same models as judges in anonymous and labelled modes.
3. Compare model judgements against human ratings to analyse bias, disagreement, and taste.

This notebook is the main project artefact. Code, outputs, charts, and commentary will live together here.

## 1. Setup and configuration

This section loads the Python packages, checks API keys, and imports helper functions from `src/`.

Run these cells once at the start of each notebook session.

In [1]:
import json
import os
import sys
from pathlib import Path

import anthropic
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm.notebook import tqdm
from xai_sdk import Client
from xai_sdk.chat import user

# Make sure the notebook can import from the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/danieljohnson/Desktop/modeleval


In [2]:
# Load API keys from .env
load_dotenv(PROJECT_ROOT / ".env")

required_keys = [
    "ANTHROPIC_API_KEY",
    "OPENAI_API_KEY",
    "XAI_API_KEY",
]

missing_keys = [key for key in required_keys if not os.getenv(key)]

if missing_keys:
    raise ValueError(f"Missing API keys: {missing_keys}")

print("All API keys loaded OK")

All API keys loaded OK


In [3]:
from src.models import call_opus, call_haiku, call_gpt5, call_grok

print("Model wrapper imports OK")

Model wrapper imports OK


### Smoke test

This cell sends a tiny prompt to all four models. It proves that the API keys, SDKs, and model wrappers work before we spend money on the real generation run.

In [4]:
test_prompt = "Write one line about ATP."

model_calls = {
    "Claude Opus 4.7": call_opus,
    "Claude Haiku 4.5": call_haiku,
    "GPT-5.5": call_gpt5,
    "Grok 4.3": call_grok,
}

for model_name, call_model in model_calls.items():
    print(f"\n--- {model_name} ---")
    response = call_model(test_prompt, max_tokens=500)
    print(response)


--- Claude Opus 4.7 ---
ATP (adenosine triphosphate) is the primary energy-carrying molecule used by cells to power biological processes.

--- Claude Haiku 4.5 ---
ATP (adenosine triphosphate) is the cell's primary energy currency, releasing energy when its high-energy phosphate bonds are broken to power cellular processes.

--- GPT-5.5 ---
ATP (adenosine triphosphate) is the primary energy-carrying molecule that powers many cellular processes.

--- Grok 4.3 ---
ATP (adenosine triphosphate) is the primary energy currency that powers cellular processes through hydrolysis of its high-energy phosphate bonds.


## 2. The eval set

This section loads the 14 prompt eval set.

The prompts are split into four categories:

1. Genre x biotech topic
2. Battle / diss tracks
3. Tribute songs
4. Storytelling tracks

The point is not just to test whether models can write lyrics. The point is to force tradeoffs between genre fidelity, scientific accuracy, lyrical craft, cleverness, and commitment.

In [ ]:
prompts_path = PROJECT_ROOT / "prompts" / "prompts.json"

with prompts_path.open("r", encoding="utf-8") as f:
    prompts = json.load(f)

prompts_df = pd.DataFrame(prompts)

print(f"Loaded {len(prompts_df)} prompts")
prompts_df